In [1]:
import archs4py as a4
import pandas as pd
import numpy as np

#path to file
file = "./data/human_tpm_v2.latest.h5"
raw_counts_file = './data/human_gene_v2.latest.h5'

In [2]:
meta_meta = a4.meta.meta(raw_counts_file, "liver", meta_fields=["title", "characteristics_ch1", "source_name_ch1", 'library_source', 'library_strategy', 'molecule_ch1'])

100%|██████████| 6/6 [00:04<00:00,  1.38it/s]


In [3]:
#filter on characterristics column as consistently the most extensive metadata 

#take only samples that collected all RNA
meta_subset = meta_meta[meta_meta.molecule_ch1 == 'total RNA']

#remove cell lines (plural often used)
meta_subset = meta_subset[~meta_subset.characteristics_ch1.str.contains('cell line')]
meta_subset = meta_subset[~meta_subset.characteristics_ch1.str.contains('cell lines')]

#remove cancer and diseased samples - removed some samples that said 'healthy/disease status: healthy' but too hard to filter for this in all cases
meta_subset = meta_subset[~meta_subset.characteristics_ch1.str.contains('cancer')]
meta_subset = meta_subset[~meta_subset.characteristics_ch1.str.contains('disease')]
meta_subset = meta_subset[~meta_subset.characteristics_ch1.str.contains('tumor')]

#wanted just hepatocytes - not endothelial cell samples
meta_subset = meta_subset[~meta_subset.characteristics_ch1.str.contains('endothelial')]
meta_subset = meta_subset[~meta_subset.characteristics_ch1.str.contains('developmental')]

#too many samples of healty/normal liver didn't specify this in their metadata, just didn't specify a disease
#meta_subset = meta_subset[meta_subset['characteristics_ch1'].str.contains('normal|healthy|Healthy')]

In [4]:
meta_subset_IDs = list(meta_subset.index)

In [5]:
sample_counts = a4.data.samples(file, meta_subset_IDs)

100%|██████████| 9541/9541 [10:21<00:00, 15.34it/s]


In [6]:
raw_counts = a4.data.samples(raw_counts_file, meta_subset_IDs)

 29%|██▊       | 2887/10054 [00:34<01:15, 95.22it/s] 

100%|██████████| 10054/10054 [01:56<00:00, 86.55it/s]


In [7]:
import mygene
mg = mygene.MyGeneInfo()
ensembl_ids = list(sample_counts.index)

In [ ]:
results = mg.querymany(
        ensembl_ids,
        scopes="'ensembl.transcript'",
        fields="symbol,name",
        species='human',
        as_dataframe=True,
        returnall=False,
    )

In [ ]:
#filters genes that don't have at least 'readThreshold' reads in 'sampleThreshold' proportion of samples - kept it at default and use only raw counts for the filtering
#filtered_exp = a4.utils.filter_genes(sample_counts, readThreshold=50, sampleThreshold=0.02, deterministic=True, aggregate=True)
filtered_raw = a4.utils.filter_genes(raw_counts, readThreshold=50, sampleThreshold=0.02, deterministic=True, aggregate=True)

In [ ]:
agg_exp = a4.utils.aggregate_duplicate_genes(filtered_exp)
agg_raw = a4.utils.aggregate_duplicate_genes(filtered_raw)

In [24]:
agg_exp = agg_exp.T
agg_raw = agg_raw.T

In [25]:
agg_exp

,ENST00000000233,ENST00000000412,ENST00000001008,ENST00000002165,ENST00000003100,ENST00000004103,ENST00000004982,ENST00000005178,ENST00000005257,ENST00000005260,...,ENST00000702574,ENST00000702588,ENST00000702614,ENST00000702686,ENST00000702693,ENST00000702921,ENST00000703064,ENST00000703176,ENST00000703212,ENST00000703309
GSM1101972,0,11,8,5,10,1,14,31,3,0,...,0,0,7,12,0,0,13,0,1,4
GSM1377536,97,46,33,70,101,133,1,15,16,36,...,204,0,0,1,0,6,4,26,0,15
GSM1377537,99,51,31,68,108,117,2,7,19,45,...,260,0,0,2,0,18,5,39,0,14
GSM1416804,34,13,7,13,39,228,8,45,4,8,...,61,0,0,2,0,0,0,0,0,8
GSM1427132,97,112,77,36,81,0,0,1,14,0,...,39,9,0,0,1,0,0,0,0,28
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GSM8800208,1,0,8,1,1,0,0,0,4,0,...,52,0,1241,0,0,0,13,0,0,2
GSM8800209,1,0,8,1,1,0,0,0,3,0,...,55,0,1483,0,0,0,12,0,0,1
GSM8800219,11,3,5,4,0,0,0,0,5,0,...,37,0,3155,0,0,0,3,0,0,1
GSM8800220,0,0,3,2,1,0,0,0,7,0,...,16,0,3199,0,0,0,0,0,0,6


In [26]:
agg_raw

,A1BG,A1BG-AS1,A1CF,A2M,A2M-AS1,A2ML1,A2MP1,A4GALT,AAAS,AACS,...,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3
GSM1101972,32,35,17,11998,48,53,26,63,77,344,...,70,2,167,223,190,11,1430,162,1127,1551
GSM1377536,4865,53,33874,51407,45,12,29,209,2331,2261,...,525,444,339,5885,2455,635,6513,22082,6679,3179
GSM1377537,5490,110,18992,34703,47,15,62,334,1821,2566,...,436,365,339,7862,2473,683,5048,31492,5325,2584
GSM1416804,1926,25,1227,7477,15,1,1,12,106,32,...,18,19,16,402,109,5,196,368,163,108
GSM1427132,17,12,0,6,1,2,0,156,1838,1367,...,1167,3264,43,239,845,7,783,2754,1633,1094
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GSM9279098,6743,49,20841,47068,288,19,14,15,227,225,...,89,11,101,705,753,193,1818,224,1172,971
GSM9279101,3644,88,11698,34633,107,15,106,20,271,232,...,71,11,78,493,735,66,1442,280,953,757
GSM9279102,6418,72,17178,50591,74,28,88,21,276,247,...,72,49,65,559,643,97,4346,356,1384,964
GSM9279103,3806,96,12376,96067,66,21,21,14,330,278,...,83,6,48,782,640,50,1773,526,1276,929


In [27]:
from Ensembl_converter import EnsemblConverter

In [10]:
agg_exp = agg_exp[agg_exp.sum(axis=1) >= 100000]
agg_exp.head(n = 5)

#make sure have same raw and log counts
agg_raw = agg_raw[agg_exp.columns]
agg_raw = agg_raw.loc[agg_exp.index]

KeyError: "None of [Index(['ENST00000000233', 'ENST00000000412', 'ENST00000001008',\n       'ENST00000002165', 'ENST00000003100', 'ENST00000004103',\n       'ENST00000004982', 'ENST00000005178', 'ENST00000005257',\n       'ENST00000005260',\n       ...\n       'ENST00000702574', 'ENST00000702588', 'ENST00000702614',\n       'ENST00000702686', 'ENST00000702693', 'ENST00000702921',\n       'ENST00000703064', 'ENST00000703176', 'ENST00000703212',\n       'ENST00000703309'],\n      dtype='str', length=10386)] are in the [columns]"

In [ ]:
agg_raw

,A1BG,A1BG-AS1,A1CF,A2M,A2M-AS1,A2ML1,A2MP1,A4GALT,AAAS,AACS,...,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3
GSM1101972,32,35,17,11998,48,53,26,63,77,344,...,70,2,167,223,190,11,1430,162,1127,1551
GSM1377536,4865,53,33874,51407,45,12,29,209,2331,2261,...,525,444,339,5885,2455,635,6513,22082,6679,3179
GSM1377537,5490,110,18992,34703,47,15,62,334,1821,2566,...,436,365,339,7862,2473,683,5048,31492,5325,2584
GSM1416804,1926,25,1227,7477,15,1,1,12,106,32,...,18,19,16,402,109,5,196,368,163,108
GSM1427132,17,12,0,6,1,2,0,156,1838,1367,...,1167,3264,43,239,845,7,783,2754,1633,1094
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GSM9279098,6743,49,20841,47068,288,19,14,15,227,225,...,89,11,101,705,753,193,1818,224,1172,971
GSM9279101,3644,88,11698,34633,107,15,106,20,271,232,...,71,11,78,493,735,66,1442,280,953,757
GSM9279102,6418,72,17178,50591,74,28,88,21,276,247,...,72,49,65,559,643,97,4346,356,1384,964
GSM9279103,3806,96,12376,96067,66,21,21,14,330,278,...,83,6,48,782,640,50,1773,526,1276,929


In [ ]:
agg_exp

,A1BG,A1BG-AS1,A1CF,A2M,A2M-AS1,A2ML1,A2MP1,A4GALT,AAAS,AACS,...,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3
GSM1101972,32,35,17,11998,48,53,26,63,77,344,...,70,2,167,223,190,11,1430,162,1127,1551
GSM1377536,4865,53,33874,51407,45,12,29,209,2331,2261,...,525,444,339,5885,2455,635,6513,22082,6679,3179
GSM1377537,5490,110,18992,34703,47,15,62,334,1821,2566,...,436,365,339,7862,2473,683,5048,31492,5325,2584
GSM1416804,1926,25,1227,7477,15,1,1,12,106,32,...,18,19,16,402,109,5,196,368,163,108
GSM1427132,17,12,0,6,1,2,0,156,1838,1367,...,1167,3264,43,239,845,7,783,2754,1633,1094
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GSM9279098,6743,49,20841,47068,288,19,14,15,227,225,...,89,11,101,705,753,193,1818,224,1172,971
GSM9279101,3644,88,11698,34633,107,15,106,20,271,232,...,71,11,78,493,735,66,1442,280,953,757
GSM9279102,6418,72,17178,50591,74,28,88,21,276,247,...,72,49,65,559,643,97,4346,356,1384,964
GSM9279103,3806,96,12376,96067,66,21,21,14,330,278,...,83,6,48,782,640,50,1773,526,1276,929


In [ ]:
agg_exp = np.log10(agg_exp + 1)

In [ ]:
agg_exp

,A1BG,A1BG-AS1,A1CF,A2M,A2M-AS1,A2ML1,A2MP1,A4GALT,AAAS,AACS,...,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3
GSM1101972,1.518514,1.556303,1.255273,4.079145,1.690196,1.732394,1.431364,1.806180,1.892095,2.537819,...,1.851258,0.477121,2.225309,2.350248,2.281033,1.079181,3.155640,2.212188,3.052309,3.190892
GSM1377536,3.687172,1.732394,4.529879,4.711031,1.662758,1.113943,1.477121,2.322219,3.367729,3.354493,...,2.720986,2.648360,2.531479,3.769820,3.390228,2.803457,3.813848,4.344058,3.824776,3.502427
GSM1377537,3.739651,2.045323,4.278594,4.540380,1.681241,1.204120,1.799341,2.525045,3.260548,3.409426,...,2.640481,2.563481,2.531479,3.895588,3.393400,2.835056,3.703205,4.498214,3.726401,3.412461
GSM1416804,3.284882,1.414973,3.089198,3.873785,1.204120,0.301030,0.301030,1.113943,2.029384,1.518514,...,1.278754,1.301030,1.230449,2.605305,2.041393,0.778151,2.294466,2.567026,2.214844,2.037426
GSM1427132,1.255273,1.113943,0.000000,0.845098,0.301030,0.477121,0.000000,2.195900,3.264582,3.136086,...,3.067443,3.513883,1.643453,2.380211,2.927370,0.903090,2.894316,3.440122,3.213252,3.039414
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GSM9279098,3.828918,1.698970,4.318939,4.672735,2.460898,1.301030,1.176091,1.204120,2.357935,2.354108,...,1.954243,1.079181,2.008600,2.848805,2.877371,2.287802,3.259833,2.352183,3.069298,2.987666
GSM9279101,3.561698,1.949390,4.068149,4.539503,2.033424,1.204120,2.029384,1.322219,2.434569,2.367356,...,1.857332,1.079181,1.897627,2.693727,2.866878,1.826075,3.159266,2.448706,2.979548,2.879669
GSM9279102,3.807467,1.863323,4.234998,4.704082,1.875061,1.462398,1.949390,1.342423,2.442480,2.394452,...,1.863323,1.698970,1.819544,2.748188,2.808886,1.991226,3.638190,2.552668,3.141450,2.984527
GSM9279103,3.580583,1.986772,4.092615,4.982579,1.826075,1.342423,1.342423,1.176091,2.519828,2.445604,...,1.924279,0.845098,1.690196,2.893762,2.806858,1.707570,3.248954,2.721811,3.106191,2.968483


In [ ]:
agg_raw

In [ ]:
DATA_ROOT = '../data/Full data files'
agg_exp.to_csv(f'{DATA_ROOT}/ARCHS4_healthy_log.tsv', sep = '\t')
agg_raw.to_csv(f'{DATA_ROOT}/ARCHS4_healthy_RAW.tsv', sep = '\t')

print(f'Raw counts shape is {agg_raw.shape}')
print(f'log counts shape is {agg_exp.shape}')

Raw counts shape is (9992, 29617)
log counts shape is (9992, 29617)


In [ ]:
#refilter meta meta to samples subsequently removed by total count filtering
meta_subset = meta_subset.loc[agg_exp.index]

In [ ]:
female_subset = meta_subset[meta_subset.characteristics_ch1.str.contains('female')]
male_subset = meta_subset[meta_subset.characteristics_ch1.str.contains('male')]
#all 'female' samples will be in 'male' - keep only males sampels that are not in female samples
male_subset = male_subset[~male_subset.index.isin(list(female_subset.index))]

In [ ]:
print(male_subset.shape, female_subset.shape)

(1160, 6) (252, 6)


In [ ]:
female_subset.to_csv(f'{DATA_ROOT}/ARCHS4_female_healthy_meta.csv')
male_subset.to_csv(f'{DATA_ROOT}/ARCHS4_male_healthy_meta.csv')
meta_subset.to_csv(f'{DATA_ROOT}/ARCHS4_all_healthy_meta.csv')